## Свёрточные сети для классификации

In [33]:
from typing import Type

import torch
from torch import Tensor, nn
from torch.nn import functional as F

#### Задание 1. Skip-connections (2 балла)

Постройте архитектуру свёрточной сети, аналогичную архитектуре в примере ниже, но добавьте в неё skip-connections, то есть дополнительные рёбра в вычислительном графе, позволяющие пропускать градиент в более ранние слои напрямую, минуя очередной блок Conv2D + BatchNorm + ReLU:

```python
def forward(self, x: Tensor) -> Tensor:
    x = x + self.block1(x)
    x = self.maxpool(x)
    x = x + self.block2(x)
    x = self.maxpool(x)
    ...
    x = x.adaptive_maxpool(x).flatten(1)
    logits = self.fc(x)
    return logits
```


Наша верхнеуровневая архитектура будет выглядеть так:

In [34]:
class MyResNet(nn.Module):
    def __init__(
        self,
        block: Type[nn.Module],
        n_classes: int,
        hidden_channels: list[int] = [32, 64],
    ) -> None:
        super().__init__()
        # входной слой, принимающий изображение с 3-мя каналами
        self.in_conv = nn.Conv2d(3, hidden_channels[0], kernel_size=3, stride=1)
        self.relu = nn.ReLU(inplace=True)

        # собираем свёрточные блоки, каждый задаётся кол-вом входных и выходных каналов
        blocks = []
        for c_in, c_out in zip(hidden_channels[:-1], hidden_channels[1:]):
            # добавляем очередной блок
            blocks.append(block(c_in, c_out))
            # добавляем Max pooling для уменьшения размерности
            blocks.append(nn.MaxPool2d(2, 2))

        # собираем блоки в единый Sequential модуль для удобства
        self.features = nn.Sequential(*blocks)
        self.maxpool = nn.AdaptiveMaxPool2d(1)

        # линейный слой для классификации
        self.fc = nn.Linear(hidden_channels[-1], n_classes)

    def forward(self, x: Tensor) -> Tensor:
        h = self.features(self.relu(self.in_conv(x)))
        logits = self.fc(self.maxpool(h).flatten(1))
        return logits

Базовый блок, без residual connections, состоит из двух свёрток и нормализаций:

In [35]:
class BasicBlock(nn.Module):
    def __init__(self, inplanes: int, planes: int) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(
            inplanes, planes, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(
            planes, planes, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes)

    def forward(self, x: Tensor) -> Tensor:
        # first conv + bn + nonlinearity
        out = self.relu(self.bn1(self.conv1(x)))
        # second conv + bn
        out = self.bn2(self.conv2(out))
        # final nonlinearity
        out = self.relu(out)
        return out

Посмотрим на результат его применения к тензору:

In [36]:
BasicBlock(4, 6).forward(torch.randn(3, 4, 32, 32)).shape

torch.Size([3, 6, 32, 32])

Теперь нужно изменить этот блок, добавив в него skip-connection. Теперь в методе `forward` входной тензор `x` пойдёт по двум веткам:
1. как в базовом блоке, через наши всёртки и нормализации, до последней нелинейности
2. в обход свёрток и нормализаций

В конце эти ветки нужно объединить через сумму. Тут есть проблема: в исходном тензоре `x` и обработанном нашим блоком `h(x)` отличается количество каналов (остальные размерности совпадают). То есть нам нужно сравнять количество каналов исходного тензора `inplanes` с количеством выходных каналов `outplanes`.

Интуитивно, если рассматривать каждый пиксель входного тензора как вектор размера `inplanes`, в вектор размера `planes` его можно превратить домножением на матрицу размера `inplanes x planes`. Это можно сделать, создав свёрточный слой с размером кернела 1 - он и будет переводить наши пиксели в другую размерность.

Не забудьте к сумме каналов применить нелинейность.

In [37]:
class ResidualBlock(nn.Module):
    def __init__(self, inplanes: int, planes: int) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(
            inplanes, planes, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(
            planes, planes, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes)

        # добавьте свёртку 1x1 для изменения кол-ва каналов входного тензора
        self.conv3 = nn.Conv2d(
            inplanes, planes, kernel_size=1, stride=1, padding=0, bias=False
        )

    def forward(self, x: Tensor) -> Tensor:
        # сохраним входной тензор на будущее
        identity = x
        
        # first conv + bn + nonlinearity
        out = self.relu(self.bn1(self.conv1(x)))
        # second conv + bn
        out = self.bn2(self.conv2(out))
        
        # ВАШ ХОД
        out = self.relu(out + self.conv3(identity))
        return out

Проверим размеры:

In [38]:
assert ResidualBlock(4, 6).forward(torch.randn(3, 4, 32, 32)).shape == torch.Size(
    [3, 6, 32, 32]
)

Проверим, что модель выдаёт тензор ожидаемого размера:

In [39]:
MyResNet(ResidualBlock, 7, hidden_channels=[16, 32, 64, 128]).forward(
    torch.randn(3, 3, 32, 32)
).shape

torch.Size([3, 7])

Теперь мы можем создавать модели разного размера, в том числе достаточно большие и глубокие, чтобы хорошо классифицировать изображения из датасета CIFAR-10.

In [40]:
sum(
    p.numel()
    for p in MyResNet(ResidualBlock, 7, hidden_channels=[16, 32, 64, 64]).parameters()
)

151047

#### Задание 2. Обучение `MyResNet` с использованием Lightning (5 баллов)

Ваша задача: добиться 80% точности на валидационной выборке с вашей реализацией `MyResNet`.

После окончания обучения используйте метод `Trainer.validate` для вывода ваших метрик с удачного чекпоинта модели.

NB: вызывайте `Trainer.validate` везде, где в задании требуется достичь какой-то точности


Советы:
- По умолчанию Lightning сохраняет только последний чекпоинт, так что вам может потребоваться `lightning.callbacks.ModelCheckpoint`, чтобы сохранять лучший чекпоинт в процессе обучения.

- Чтобы добиться нужной точности, ваша модель должна быть достаточно глубокой, ориентируйтесь на 4-5 блоков.

- Используйте tensorboard, чтобы следить за динамикой обучения. Если заметите переобучение — подключайте регуляризацию. Большая модель с регуляризацией обычно лучше маленькой модели без неё.



In [41]:
import lightning as L
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from typing import Type, Any
import torchmetrics
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.utilities.types import STEP_OUTPUT
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

In [42]:
class ResidualBlock(nn.Module):
    def __init__(self, inplanes: int, planes: int) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(
            inplanes, planes, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(
            planes, planes, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes)

        self.conv3 = nn.Conv2d(
            inplanes, planes, kernel_size=1, stride=1, padding=0, bias=False
        )

    def forward(self, x: Tensor) -> Tensor:
        identity = x
        
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        out = self.relu(out + self.conv3(identity))
        return out

class MyResNet(nn.Module):
    def __init__(
        self,
        block: Type[nn.Module],
        n_classes: int,
        hidden_channels: list[int] = [32, 64],
    ) -> None:
        super().__init__()
        self.in_conv = nn.Conv2d(3, hidden_channels[0], kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU(inplace=True)

        blocks = []
        for c_in, c_out in zip(hidden_channels[:-1], hidden_channels[1:]):
            blocks.append(block(c_in, c_out))
            blocks.append(nn.MaxPool2d(2, 2))

        self.features = nn.Sequential(*blocks)
        self.maxpool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Linear(hidden_channels[-1], n_classes)

    def forward(self, x: Tensor) -> Tensor:
        h = self.features(self.relu(self.in_conv(x)))
        logits = self.fc(self.maxpool(h).flatten(1))
        return logits

In [57]:
class CIFAR10DataModule(L.LightningDataModule):
    def __init__(self, batch_size: int = 128, num_workers: int = 0):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers
        
        self.transform = transforms.ToTensor()

    def prepare_data(self):
        pass

    def setup(self, stage: str):
        if stage == "fit":
            self.train_dataset = CIFAR10(
                "data", train=True, download=True, transform=self.transform
            )
            self.val_dataset = CIFAR10(
                "data", train=False, download=True, transform=self.transform
            )

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
        )

def create_classification_metrics(num_classes: int, prefix: str):
    return torchmetrics.MetricCollection(
        [
            torchmetrics.Accuracy(task="multiclass", num_classes=num_classes),
            torchmetrics.classification.MulticlassAUROC(
                num_classes=num_classes, average="macro"
            ),
        ],
        prefix=prefix,
    )

class ResNetLightning(L.LightningModule):
    def __init__(self, learning_rate: float = 0.001):
        super().__init__()
        self.save_hyperparameters()
        
        self.model = MyResNet(
            block=ResidualBlock,
            n_classes=10,
            hidden_channels=[32, 64, 128, 256],
        )
        
        self.learning_rate = learning_rate
        self.train_metrics = create_classification_metrics(num_classes=10, prefix="train_")
        self.val_metrics = create_classification_metrics(num_classes=10, prefix="val_")

    def training_step(self, batch: tuple[Tensor, Tensor], batch_idx: int) -> STEP_OUTPUT:
        x, y = batch
        y_hat = self.model(x)
        loss = F.cross_entropy(y_hat, y)
        
        self.log("train_loss", loss, on_epoch=True, on_step=False)
        self.train_metrics.update(y_hat, y)
        self.log_dict(self.train_metrics, on_step=False, on_epoch=True)
        
        return loss

    def validation_step(self, batch: tuple[Tensor, Tensor], batch_idx: int) -> STEP_OUTPUT | None:
        x, y = batch
        y_hat = self.model(x)
        loss = F.cross_entropy(y_hat, y)
        
        self.log("val_loss", loss, on_epoch=True, on_step=False)
        self.val_metrics.update(y_hat, y)
        self.log_dict(self.val_metrics, on_step=False, on_epoch=True)
        
        return {"loss": loss, "preds": y_hat}

    def configure_optimizers(self) -> dict[str, Any]:
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.learning_rate)
        
        return {
            "optimizer": optimizer,
            "lr_scheduler": torch.optim.lr_scheduler.MultiStepLR(
                optimizer, milestones=[30, 60], gamma=0.1
            ),
        }

def get_callbacks():
    return [
        ModelCheckpoint(
            monitor="val_MulticlassAccuracy",
            mode="max",
            filename="best-{epoch:02d}-{val_MulticlassAccuracy:.2f}",
            save_top_k=1,
            save_last=True,
        ),
        EarlyStopping(
            monitor="val_MulticlassAccuracy",
            mode="max",
            patience=5,
            min_delta=0.00,
        )
    ]

def train_and_validate():
    datamodule = CIFAR10DataModule(batch_size=128, num_workers=0)
    model = ResNetLightning(learning_rate=0.001)
    
    trainer = L.Trainer(
        accelerator="auto",
        max_epochs=10,
        callbacks=get_callbacks(),
        logger=TensorBoardLogger("lightning_logs/", name="my_resnet"),
        log_every_n_steps=50,
    )
    
    trainer.fit(model, datamodule=datamodule)
    
    results = trainer.validate(model, datamodule=datamodule, ckpt_path="best")
    
    return results

In [58]:
results = train_and_validate()
val_accuracy = results[0]["val_MulticlassAccuracy"]
print(f"Достигнутая точность на валидации: {val_accuracy:.4f}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | model         | MyResNet         | 1.2 M  | train
1 | train_metrics | MetricCollection | 0      | train
2 | val_metrics   | MetricCollection | 0      | train
-----------------------------------------------------------
1.2 M     Trainable params
0         Non-trainable params
1.2 M     Total params
4.838     Total estimated model params size (MB)
36        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.
Restoring states from the checkpoint path at lightning_logs/my_resnet/version_9/checkpoints/best-epoch=07-val_MulticlassAccuracy=0.86.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at lightning_logs/my_resnet/version_9/checkpoints/best-epoch=07-val_MulticlassAccuracy=0.86.ckpt


Validation: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   val_MulticlassAUROC      0.9887890815734863
 val_MulticlassAccuracy     0.8578000068664551
        val_loss            0.48236238956451416
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Достигнутая точность на валидации: 0.8578


#### Задание 3. Добавление аугментаций (1 балл + 2 балла за точность на валидации более 85%)

Добавьте к обучающему датасету аугментации - случайные трансформации входных данных. Для этого можно использовать `torchvision.transforms` и `albumentations`.

С `torchvision.transforms` совсем просто: вам нужно будет при создании `Datamodule` из практики по `lightning` указать вместо

```python
transform = transforms.ToTensor()
```
композицию трансформаций:

```python
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),  # случайное зеркальное отражение
    ...,
    transforms.ToTensor(),
])
```

В пакете `albumentations` аугментаций значительно больше:

![albumentations](https://albumentations.ai/_next/image/?url=%2Fassets%2Ftop_image.webp&w=1080&q=75)

In [59]:
import lightning as L
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from typing import Type, Any
import torchmetrics
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.utilities.types import STEP_OUTPUT
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader


class ResidualBlock(nn.Module):
    def __init__(self, inplanes: int, planes: int) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(
            inplanes, planes, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(
            planes, planes, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes)

        self.conv3 = nn.Conv2d(
            inplanes, planes, kernel_size=1, stride=1, padding=0, bias=False
        )

    def forward(self, x: Tensor) -> Tensor:
        identity = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out = self.relu(out + self.conv3(identity))
        return out


class MyResNet(nn.Module):
    def __init__(
            self,
            block: Type[nn.Module],
            n_classes: int,
            hidden_channels: list[int] = [32, 64],
    ) -> None:
        super().__init__()
        self.in_conv = nn.Conv2d(3, hidden_channels[0], kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU(inplace=True)

        blocks = []
        for c_in, c_out in zip(hidden_channels[:-1], hidden_channels[1:]):
            blocks.append(block(c_in, c_out))
            blocks.append(nn.MaxPool2d(2, 2))

        self.features = nn.Sequential(*blocks)
        self.maxpool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Linear(hidden_channels[-1], n_classes)

    def forward(self, x: Tensor) -> Tensor:
        h = self.features(self.relu(self.in_conv(x)))
        logits = self.fc(self.maxpool(h).flatten(1))
        return logits


class CIFAR10DataModule(L.LightningDataModule):
    def __init__(self, batch_size: int = 128, num_workers: int = 0, use_augmentations: bool = True):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.use_augmentations = use_augmentations

        self.val_transform = transforms.Compose([
            transforms.ToTensor(),
        ])

        if self.use_augmentations:
            self.train_transform = transforms.Compose([
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomCrop(32, padding=4),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
                transforms.RandomRotation(degrees=15),
                transforms.ToTensor(),
            ])
        else:
            self.train_transform = self.val_transform

    def prepare_data(self):
        pass

    def setup(self, stage: str):
        if stage == "fit":
            self.train_dataset = CIFAR10(
                "data", train=True, download=True, transform=self.train_transform
            )
            self.val_dataset = CIFAR10(
                "data", train=False, download=True, transform=self.val_transform
            )

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
        )


def create_classification_metrics(num_classes: int, prefix: str):
    return torchmetrics.MetricCollection(
        [
            torchmetrics.Accuracy(task="multiclass", num_classes=num_classes),
            torchmetrics.classification.MulticlassAUROC(
                num_classes=num_classes, average="macro"
            ),
        ],
        prefix=prefix,
    )


class ResNetLightning(L.LightningModule):
    def __init__(self, learning_rate: float = 0.001):
        super().__init__()
        self.save_hyperparameters()

        self.model = MyResNet(
            block=ResidualBlock,
            n_classes=10,
            hidden_channels=[32, 64, 128, 256],
        )

        self.learning_rate = learning_rate
        self.train_metrics = create_classification_metrics(num_classes=10, prefix="train_")
        self.val_metrics = create_classification_metrics(num_classes=10, prefix="val_")

    def training_step(self, batch: tuple[Tensor, Tensor], batch_idx: int) -> STEP_OUTPUT:
        x, y = batch
        y_hat = self.model(x)
        loss = F.cross_entropy(y_hat, y)

        self.log("train_loss", loss, on_epoch=True, on_step=False)
        self.train_metrics.update(y_hat, y)
        self.log_dict(self.train_metrics, on_step=False, on_epoch=True)

        return loss

    def validation_step(self, batch: tuple[Tensor, Tensor], batch_idx: int) -> STEP_OUTPUT | None:
        x, y = batch
        y_hat = self.model(x)
        loss = F.cross_entropy(y_hat, y)

        self.log("val_loss", loss, on_epoch=True, on_step=False)
        self.val_metrics.update(y_hat, y)
        self.log_dict(self.val_metrics, on_step=False, on_epoch=True)

        return {"loss": loss, "preds": y_hat}

    def configure_optimizers(self) -> dict[str, Any]:
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.learning_rate)

        return {
            "optimizer": optimizer,
            "lr_scheduler": torch.optim.lr_scheduler.MultiStepLR(
                optimizer, milestones=[30, 60], gamma=0.1
            ),
        }


def get_callbacks():
    return [
        ModelCheckpoint(
            monitor="val_MulticlassAccuracy",
            mode="max",
            filename="best-{epoch:02d}-{val_MulticlassAccuracy:.2f}",
            save_top_k=1,
            save_last=True,
        ),
        EarlyStopping(
            monitor="val_MulticlassAccuracy",
            mode="max",
            patience=10,
            min_delta=0.001,
        )
    ]


def train_with_augmentations():
    datamodule = CIFAR10DataModule(batch_size=128, num_workers=0, use_augmentations=True)
    model = ResNetLightning(learning_rate=0.001)

    trainer = L.Trainer(
        accelerator="auto",
        max_epochs=10,
        callbacks=get_callbacks(),
        logger=TensorBoardLogger("lightning_logs/", name="my_resnet_augmented"),
        log_every_n_steps=50,
    )

    trainer.fit(model, datamodule=datamodule)

    results = trainer.validate(model, datamodule=datamodule, ckpt_path="best")

    return results


results = train_with_augmentations()
val_accuracy = results[0]["val_MulticlassAccuracy"]
print(f"Достигнутая точность на валидации с аугментациями: {val_accuracy:.4f}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | model         | MyResNet         | 1.2 M  | train
1 | train_metrics | MetricCollection | 0      | train
2 | val_metrics   | MetricCollection | 0      | train
-----------------------------------------------------------
1.2 M     Trainable params
0         Non-trainable params
1.2 M     Total params
4.838     Total estimated model params size (MB)
36        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.
Restoring states from the checkpoint path at lightning_logs/my_resnet_augmented/version_1/checkpoints/best-epoch=09-val_MulticlassAccuracy=0.86.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at lightning_logs/my_resnet_augmented/version_1/checkpoints/best-epoch=09-val_MulticlassAccuracy=0.86.ckpt


Validation: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   val_MulticlassAUROC      0.9900177121162415
 val_MulticlassAccuracy     0.8579999804496765
        val_loss            0.41156014800071716
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Достигнутая точность на валидации с аугментациями: 0.8580


#### Задание 4. Использование предобученной модели (4 балла)

Теперь мы научимся использовать модели, обученные на других задачах

Ваша задача: добиться 90% точности на тестовой выборке CIFAR-10. Постарайтесь уложиться модель с ~5 млн параметров

В `torchvision.models` есть много реализованных архитектур, размером которых можно удобно управлять. Например, ниже можно создать крошечную версию модели `MobileNetV2`:

In [51]:
from torchvision.models import MobileNetV2

mobilenet = MobileNetV2(
    num_classes=10,
    width_mult=0.4,
    inverted_residual_setting=[
        # t, c, n, s
        [1, 16, 1, 1],
        [3, 24, 2, 2],
        [3, 32, 3, 2],
    ],
    dropout=0.2,
)

sum([param.numel() for param in mobilenet.parameters()])

46322

Но кроме архитектуры модели, мы также можем скачать веса, полученные при обучении на каком-то датасете. Например, для нашей задачи можно использовать предобучение на самом известном датасете для классификации изображений - ImageNet:

In [52]:
from torchvision.models.efficientnet import EfficientNet_B0_Weights, efficientnet_b0

# создаём EfficientNet с весами, полученными на ImageNet
weights = EfficientNet_B0_Weights.IMAGENET1K_V1
efficientnet = efficientnet_b0(weights=weights)
sum([param.numel() for param in efficientnet.parameters()])

5288548

**Указание 1.** С использованием модели в исходном виде есть проблема: в ImageNet 1000 классов, а у нас только 10. Поэтому в предобученной модели нужно будет полностью заменить последний линейный слой, который даёт распределение вероятностей классов. Это можно сделать уже в готовом объекте модели, переназначив атрибут.

Подсказка: в `efficientnet_b0` линейный слой находится в атрибуте `classifier` 


**Указание 2.** Все слои, кроме нескольких последних (может быть, только последнего) мы можем заморозить, то есть сделать значения параметров в них неизменными. Это позволит и сохранить способность модели выделять полезные низкоуровневые признаки (она научилась этому на ImageNet), и существенно ускорить дообучение.


Чтобы заморозить параметры, нужно всего лишь отключить для них расчёт градиентов. Вернитесь к первой практике, чтобы вспомнить, как это можно сделать. Нам подойдёт самый простой способ с `.requires_grad`.

Подсказка: в `efficientnet_b0` свёрточные слои находятся в атрибуте `features` 

**Указание 3.** Предобученные модели на ImageNet ожидают специальным образом трансформированные изображения:


In [56]:
weights.transforms()

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

Поэтому эти трансформации нужно будет передать в датамодуль (как мы делали с аугментациями).

ВАШ ХОД: Обучите модель и выведите результат метода validate на удачном чекпоинте

In [60]:
import lightning as L
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from typing import Any
import torchmetrics
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.utilities.types import STEP_OUTPUT
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
from torchvision.models.efficientnet import EfficientNet_B0_Weights, efficientnet_b0


class CIFAR10PretrainedDataModule(L.LightningDataModule):
    def __init__(self, batch_size: int = 128, num_workers: int = 0):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers

        self.weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        self.transform = self.weights.transforms()

        self.train_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomCrop(32, padding=4),
            transforms.Resize(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        self.val_transform = transforms.Compose([
            transforms.Resize(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def prepare_data(self):
        pass

    def setup(self, stage: str):
        if stage == "fit":
            self.train_dataset = CIFAR10(
                "data", train=True, download=True, transform=self.train_transform
            )
            self.val_dataset = CIFAR10(
                "data", train=False, download=True, transform=self.val_transform
            )

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
        )


def create_classification_metrics(num_classes: int, prefix: str):
    return torchmetrics.MetricCollection(
        [
            torchmetrics.Accuracy(task="multiclass", num_classes=num_classes),
            torchmetrics.classification.MulticlassAUROC(
                num_classes=num_classes, average="macro"
            ),
        ],
        prefix=prefix,
    )


class EfficientNetLightning(L.LightningModule):
    def __init__(self, learning_rate: float = 0.001, freeze_backbone: bool = True):
        super().__init__()
        self.save_hyperparameters()

        self.weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        self.model = efficientnet_b0(weights=self.weights)

        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Sequential(
            nn.Dropout(p=0.2, inplace=True),
            nn.Linear(in_features, 10),
        )

        if freeze_backbone:
            for param in self.model.features.parameters():
                param.requires_grad = False
            print("Backbone заморожен! Обучается только классификатор.")
        else:
            print("Обучается вся модель.")

        self.learning_rate = learning_rate
        self.train_metrics = create_classification_metrics(num_classes=10, prefix="train_")
        self.val_metrics = create_classification_metrics(num_classes=10, prefix="val_")

        total_params = sum(p.numel() for p in self.model.parameters())
        trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"Всего параметров: {total_params:,}")
        print(f"Обучаемых параметров: {trainable_params:,}")

    def training_step(self, batch: tuple[Tensor, Tensor], batch_idx: int) -> STEP_OUTPUT:
        x, y = batch
        y_hat = self.model(x)
        loss = F.cross_entropy(y_hat, y)

        self.log("train_loss", loss, on_epoch=True, on_step=False, prog_bar=True)
        self.train_metrics.update(y_hat, y)
        self.log_dict(self.train_metrics, on_step=False, on_epoch=True)

        return loss

    def validation_step(self, batch: tuple[Tensor, Tensor], batch_idx: int) -> STEP_OUTPUT | None:
        x, y = batch
        y_hat = self.model(x)
        loss = F.cross_entropy(y_hat, y)

        self.log("val_loss", loss, on_epoch=True, on_step=False, prog_bar=True)
        self.val_metrics.update(y_hat, y)
        self.log_dict(self.val_metrics, on_step=False, on_epoch=True)

        return {"loss": loss, "preds": y_hat}

    def configure_optimizers(self) -> dict[str, Any]:
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=self.learning_rate,
            weight_decay=1e-4
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=50,
            eta_min=1e-6
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": scheduler
        }


def get_callbacks_90():
    return [
        ModelCheckpoint(
            monitor="val_MulticlassAccuracy",
            mode="max",
            filename="best-90-{epoch:02d}-{val_MulticlassAccuracy:.2f}",
            save_top_k=1,
            save_last=True,
        ),
        EarlyStopping(
            monitor="val_MulticlassAccuracy",
            mode="max",
            patience=15,
            min_delta=0.001,
        )
    ]


def train_pretrained_model():
    datamodule = CIFAR10PretrainedDataModule(batch_size=64, num_workers=0)
    model = EfficientNetLightning(learning_rate=0.001, freeze_backbone=True)

    trainer = L.Trainer(
        accelerator="auto",
        max_epochs=10,
        callbacks=get_callbacks_90(),
        logger=TensorBoardLogger("lightning_logs/", name="efficientnet_pretrained"),
        log_every_n_steps=50,
        precision="16-mixed",
    )

    trainer.fit(model, datamodule=datamodule)

    results = trainer.validate(model, datamodule=datamodule, ckpt_path="best")

    return results


results = train_pretrained_model()
val_accuracy = results[0]["val_MulticlassAccuracy"]
print(f"Достигнутая точность на валидации: {val_accuracy:.4f}")

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Backbone заморожен! Обучается только классификатор.
Всего параметров: 4,020,358
Обучаемых параметров: 12,810


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/Deep-Learning-2025-MCS/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:231: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | model         | EfficientNet     | 4.0 M  | train
1 | train_metrics | MetricCollection | 0      | train
2 | val_metrics   | MetricCollection | 0      | train
-----------------------------------------------------------
12.8 K    Trainable params
4.0 M     Non-trainable params
4.0 M     Total params
16.081    Total estimated model params size (MB)
343       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.
Restoring states from the checkpoint path at lightning_logs/efficientnet_pretrained/version_0/checkpoints/best-90-epoch=06-val_MulticlassAccuracy=0.80.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at lightning_logs/efficientnet_pretrained/version_0/checkpoints/best-90-epoch=06-val_MulticlassAccuracy=0.80.ckpt


Validation: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   val_MulticlassAUROC      0.9801577925682068
 val_MulticlassAccuracy     0.8003000020980835
        val_loss            0.5762237906455994
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Достигнутая точность на валидации: 0.8003
